In [ ]:
"""
Quick Schema Validation — Module 1 
"""

import glob
import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
# ── Environment detection ──────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset/Exports')
    BASE_DIR    = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    IN_COLAB    = True
except Exception:
    BASE_DIR    = Path.cwd().parent / 'Dataset'
    IN_COLAB    = False

EXPORTS_DIR = BASE_DIR / 'Exports'
print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Exports dir : {EXPORTS_DIR}')
print(f'Exists      : {EXPORTS_DIR.exists()}')

EXPECTED_COLUMNS = [
    "Period", "Commodity", "Province", "Country",
    "State", "Value ($)", "Quantity", "Unit of measure"
]
EXPECTED_COL_COUNT = len(EXPECTED_COLUMNS)

# ──────────────────────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────────────────────
def load_raw(filepath):
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()
    
    # Remove footer lines (metadata at bottom of file)
    if "Period" in df.columns:
        # Step 1: Remove rows where Period contains metadata keywords
        metadata_keywords = ["source:", "note:", "notes:", "data source:", "disclaimer:", 
                            "citation:", "data extracted:", "extracted:", "date:", 
                            "© ", "copyright", "confidential"]
        period_lower = df["Period"].fillna("").astype(str).str.lower().str.strip()
        mask = period_lower.str.startswith(tuple(metadata_keywords))
        df = df[~mask]
        
        # Step 2: Remove rows where Period is not a valid date format
        # Valid dates should parse correctly or be empty
        period_values = df["Period"].fillna("").astype(str).str.strip()
        
        # Identify rows that look like dates (YYYY-MM-DD or similar) or are empty
        valid_mask = (
            (period_values == "") |  # Empty is ok (will be caught as missing value)
            (period_values.str.match(r'^\d{4}-\d{2}-\d{2}')) |  # YYYY-MM-DD
            (period_values.str.match(r'^\d{4}/\d{2}/\d{2}')) |  # YYYY/MM/DD
            (period_values.str.match(r'^\d{2}-\d{2}-\d{4}')) |  # DD-MM-YYYY
            (pd.to_datetime(df["Period"], errors='coerce').notna())  # Any parseable date
        )
        df = df[valid_mask]
        
        # Step 3: Remove trailing completely empty rows
        last_valid_idx = None
        for idx in reversed(df.index):
            row = df.loc[idx]
            has_data = row.notna().any() and (row.astype(str).str.strip() != "").any()
            if has_data:
                last_valid_idx = idx
                break
        
        if last_valid_idx is not None:
            df = df.loc[:last_valid_idx]
    
    df["_source_file"] = Path(filepath).name
    return df

def load_all(files):
    frames = []
    errors = []
    for f in files:
        try:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors

# ──────────────────────────────────────────────────────────────
# CAST FOR ANALYSIS
# ──────────────────────────────────────────────────────────────
def cast(df):
    df = df.copy()
    df["_period"]   = pd.to_datetime(df.get("Period"),   errors="coerce")
    df["_value"]    = pd.to_numeric(df.get("Value ($)"), errors="coerce")
    df["_qty"]      = pd.to_numeric(df.get("Quantity"),  errors="coerce")
    return df

# ──────────────────────────────────────────────────────────────
# MODULE 1 — SCHEMA VALIDATION
# ──────────────────────────────────────────────────────────────
def module_schema(raw, load_errors):
    results = []
    examples = {}  # Store example rows for failures

    # Column presence
    actual = list(raw.columns.drop("_source_file", errors="ignore"))
    missing_cols = [c for c in EXPECTED_COLUMNS if c not in actual]
    extra_cols   = [c for c in actual if c not in EXPECTED_COLUMNS]

    results.append({
        "check": "Expected column count",
        "expected": EXPECTED_COL_COUNT,
        "found": len(actual),
        "status": "✅ OK" if len(actual) == EXPECTED_COL_COUNT else "❌ FAIL",
        "detail": f"Missing: {missing_cols}" if missing_cols else ""
    })

    results.append({
        "check": "Missing columns",
        "expected": ", ".join(EXPECTED_COLUMNS),
        "found": ", ".join(actual) if actual else "-",
        "status": "✅ OK" if not missing_cols else "❌ FAIL",
        "detail": ", ".join(missing_cols) if missing_cols else "None"
    })

    results.append({
        "check": "Extra / unexpected columns",
        "expected": "-",
        "found": ", ".join(extra_cols) if extra_cols else "None",
        "status": "⚠️ WARN" if extra_cols else "✅ OK",
        "detail": ", ".join(extra_cols) if extra_cols else "None"
    })

    # Data type checks
    df = cast(raw)
    type_checks = {
        "Period":    ("Date",   df["_period"], raw.get("Period", pd.Series())),
        "Value ($)": ("Numeric",df["_value"],  raw.get("Value ($)", pd.Series())),
        "Quantity":  ("Numeric",df["_qty"],    raw.get("Quantity",  pd.Series())),
    }

    for col, (expected_type, parsed_series, raw_series) in type_checks.items():
        # Count values that exist in raw but failed to parse
        exists_in_raw = raw_series.notna() & (raw_series.astype(str).str.strip() != "")
        parsed_ok = parsed_series.notna()
        
        # Parse failures = exists but couldn't parse
        parse_fail_mask = exists_in_raw & ~parsed_ok
        failed = parse_fail_mask.sum()
        
        status = "✅ OK" if failed == 0 else "❌ FAIL"
        detail = (f"{failed:,} non-empty values could not be parsed as {expected_type}" 
                  if failed > 0 else "All values parsed successfully")
        
        check_key = f"Data type: {col}"
        results.append({
            "check": check_key,
            "expected": expected_type,
            "found": f"{failed} parse failures" if failed > 0 else "All valid",
            "status": status,
            "detail": detail
        })
        
        # Store example rows with parse failures
        if failed > 0:
            fail_rows = raw[parse_fail_mask].head(5)
            display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                           if c in fail_rows.columns]
            examples[check_key] = fail_rows[display_cols].copy()
    
    # Text columns
    for col in ["Province", "Country", "Commodity", "Unit of measure"]:
        results.append({
            "check": f"Data type: {col}",
            "expected": "Text",
            "found": "-",
            "status": "✅ OK",
            "detail": "Text — no type casting required"
        })

    # Mixed type check
    for col in ["Value ($)", "Quantity"]:
        if col in raw.columns:
            raw_col = raw[col]
            exists = raw_col.notna() & (raw_col.astype(str).str.strip() != "")
            coerced = pd.to_numeric(raw_col, errors="coerce")
            mixed_mask = exists & coerced.isna()
            n_fail = mixed_mask.sum()
            
            check_key = f"Mixed types: {col}"
            results.append({
                "check": check_key,
                "expected": "All numeric",
                "found": f"{n_fail:,} non-numeric strings" if n_fail > 0 else "All numeric",
                "status": "❌ FAIL" if n_fail > 0 else "✅ OK",
                "detail": f"{n_fail} cells contain text in numeric column" if n_fail > 0 else "No mixed types"
            })
            
            # Store example rows with mixed types
            if n_fail > 0:
                fail_rows = raw[mixed_mask].head(5)
                display_cols = [c for c in ["Period", "Commodity", "Province", "Country", "Value ($)", "Quantity", "_source_file"] 
                               if c in fail_rows.columns]
                examples[check_key] = fail_rows[display_cols].copy()

    # Load errors
    for e in load_errors:
        results.append({
            "check": "File load error",
            "expected": "-",
            "found": e["file"],
            "status": "❌ FAIL",
            "detail": e["error"]
        })

    return results, examples

# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(results, examples):
    print("\n" + "="*80)
    print("  MODULE 1 — SCHEMA VALIDATION")
    print("="*80 + "\n")
    
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['check']}")
        print(f"   Expected: {r['expected']}")
        print(f"   Found:    {r['found']}")
        print(f"   Status:   {r['status']}")
        if r['detail']:
            print(f"   Detail:   {r['detail']}")
        
        # Print example rows if this check failed
        if "❌" in r['status'] and r['check'] in examples:
            print(f"\n   📋 Example rows with failures (showing up to 5):")
            print("   " + "-" * 76)
            df_example = examples[r['check']]
            for idx, row in df_example.iterrows():
                print(f"   Row {idx}:")
                for col in df_example.columns:
                    val = row[col]
                    # Highlight the problematic value
                    if pd.isna(val):
                        val_str = "[EMPTY]"
                    elif str(val).strip() == "":
                        val_str = "[EMPTY]"
                    else:
                        val_str = str(val)[:50]  # Truncate long values
                    print(f"      {col:20s}: {val_str}")
                print()
            print("   " + "-" * 76)
        
        print()

# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*80)
    print("  SCHEMA VALIDATION — MODULE 1 ONLY")
    print("="*80)

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)
    print(f"\nFound {len(export_files)} CSV file(s) in {EXPORTS_DIR}")

    if not export_files:
        print("❌ No files found. Check EXPORTS_DIR path.")
        exit(1)

    print("Loading files...")
    raw, load_errors = load_all(export_files)
    print(f"Loaded {len(raw):,} rows from {len(set(raw['_source_file']))} file(s)")
    if load_errors:
        print(f"⚠️  {len(load_errors)} file(s) failed to load")

    print("\nRunning schema validation...")
    results, examples = module_schema(raw, load_errors)
    
    print_results(results, examples)
    
    # Summary
    fails = sum(1 for r in results if "❌" in r["status"])
    warns = sum(1 for r in results if "⚠️" in r["status"])
    ok    = sum(1 for r in results if "✅" in r["status"])
    
    print("="*80)
    print(f"  SUMMARY: {ok} OK  |  {warns} WARNINGS  |  {fails} FAILURES")
    print("="*80 + "\n")



  SCHEMA VALIDATION — MODULE 1 ONLY

Found 154 CSV file(s) in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
Loading files...
Loaded 2,536,465 rows from 154 file(s)

Running schema validation...

  MODULE 1 — SCHEMA VALIDATION

1. Expected column count
   Expected: 8
   Found:    8
   Status:   ✅ OK

2. Missing columns
   Expected: Period, Commodity, Province, Country, State, Value ($), Quantity, Unit of measure
   Found:    Period, Commodity, Province, Country, State, Value ($), Quantity, Unit of measure
   Status:   ✅ OK
   Detail:   None

3. Extra / unexpected columns
   Expected: -
   Found:    None
   Status:   ✅ OK
   Detail:   None

4. Data type: Period
   Expected: Date
   Found:    All valid
   Status:   ✅ OK
   Detail:   All values parsed successfully

5. Data type: Value ($)
   Expected: Numeric
   Found:    All valid
   Status:   ✅ OK
   Detail:   All values parsed successfully

6. Data type: Quantity
   Expected: Numeric
   Found:    All valid
   Status: